# INTEGRATION RANGES

In [ ]:
import sys; sys.path.insert(0, '../'); from lib import *;
figure_features()

# Set options for general visualitation
OPT  = {
    "MICRO_SEC":   True,                # Time in microseconds (True/False)
    "NORM":        False,               # Runs can be displayed normalised (True/False)
    "ALIGN":       True,                # Aligns waveforms in peaktime (True/False)
    "LOGY":        False,               # Runs can be displayed in logy (True/False)
    "SHOW_AVE":    "",                  # If computed, vis will show average (AveWvf,AveWvfSPE,etc.)
    "SHOW_PARAM":  False,               # Print terminal information (True/False)
    "CHARGE_KEY":  "ChargeAveRange",    # Select charge info to be displayed. Default: "ChargeAveRange" (if computed)
    "PEAK_FINDER": False,               # Finds possible peaks in the window (True/False)
    "LEGEND":      True,                # Shows plot legend (True/False)
    "SHOW":        True
    }


In [ ]:
info=read_input_file('MegaCell_LAr_dec');
PRESET="ALL";
runs=[60];
channels=[0];

my_runs=load_npy(runs,channels,info,preset=PRESET,debug=False) #Struct my_runs[run][channel]['Variable'][event]

In [ ]:
run=runs[0];
AnaADC={};
Mean_AnaADC={};
Lim_ped={};
for c in channels:
    AnaADC[c]=(my_runs[run][c]['PChannel']*my_runs[run][c]['RawADC'].T-my_runs[run][c]['PChannel']*my_runs[run][c]['RawPreTriggerMean'][0].T).T;
    Mean_AnaADC[c]=np.mean(AnaADC[c],axis=0);
    Lim_ped[c]=my_runs[run][c]['RawPedLim'];

In [ ]:
Charge={};
ranges=np.arange(100,1000,100);
Charge_aux=[];
c=channels[0];
for i in np.arange(len(ranges)):
    Charge_aux=np.sum(AnaADC[c][:,Lim_ped[c]+10:np.argmax(Mean_AnaADC[c])+ranges[i]],axis=1)
    Charge[i]=np.histogram(Charge_aux,bins=2000);


In [ ]:
fig1=go.Figure();
fig1.add_trace(go.Scatter(x=np.arange(len(Mean_AnaADC[c])),y=Mean_AnaADC[0],name='WaveForm Mean'))
fig1.add_vline(x=ranges[8]+np.argmax(Mean_AnaADC[c]))
fig1.add_vline(x=Lim_ped[0]+10)
fig1.update_layout(title='Run '+str(run)+' Channel '+str(c), title_x=0.5,showlegend=True,template='presentation')
fig1.update_xaxes(title='Ticks')
fig1.update_yaxes(title='ADC counts')

In [ ]:
fig1=go.Figure()
for i in np.arange(len(ranges)):
    fig1.add_trace(go.Scatter(x=Charge[i][1],y=Charge[i][0],name=str(ranges[i]*4e-3)+' us'))
fig1.update_yaxes(type='log',mirror=True,showline=True,showgrid=True,tickformat='.1s')
fig1.update_xaxes(mirror=True,showline=True,showgrid=True)
fig1.update_layout(template='presentation',title='Ranges of Integration, Run '+str(run)+' Channel '+str(c),title_x=0.5)
fig1.update_yaxes(title='Counts')
fig1.update_xaxes(title='Charge [ADC*ticks]')
fig1.show()

FFT

In [ ]:
run=runs[0];
RFFT={};
freq={};
RFFT_mean={};
c=channels[0];
event=np.random.randint(len(my_runs[run][c]["RawADC"]));

for c in channels:
    RFFT[c]=np.abs(np.fft.rfft(my_runs[run][c]["RawADC"][event]))
    freq[c]=np.fft.rfftfreq(my_runs[run][c]["RawADC"][0].size,d=my_runs[run][c]["Sampling"])
    RFFT_mean[c]=np.mean(np.abs(np.fft.rfft(my_runs[run][c]["RawADC"])),axis=0)
    #fig1.add_trace(go.Scatter(x=freq[c],y=RFFT[c],name="Channel "+str(c)+" Event="+str(event)))
    #fig2.add_trace(go.Scatter(x=freq[c],y=RFFT_mean[c],name="Channel "+str(c)+" mean"))

In [ ]:
#fig1=go.Figure()
fig2=go.Figure()
for c in channels:
    #fig1.add_trace(go.Scatter(x=freq[c],y=RFFT[c],name="Channel "+str(c)+" Event="+str(event)))
    fig2.add_trace(go.Scatter(x=freq[c],y=RFFT_mean[c],name="Channel "+str(c)+" mean"))

#fig1.update_layout(title='Noise '+str(event),title_x = 0.5, xaxis_title='Frequency [Hz]', yaxis_title='Amplitude')
#fig1.update_xaxes(mirror=True,showline=True,type="log", tickformat=".1s")
#fig1.update_yaxes(mirror=True,showline=True,type="log")
#fig1.show()
fig2.update_layout(title='Noise mean, run '+str(run), title_x = 0.5, xaxis_title='Frequency [Hz]', yaxis_title='Amplitude',template='presentation')
fig2.update_xaxes(mirror=True,showline=True,type="log", tickformat=".1s")
fig2.update_yaxes(mirror=True,showline=True,type="log",range=[2,5])
fig2.show()

In [ ]:
c=0;
# Find peaks and valleys
from sklearn.cluster import KMeans

Good_charge=np.sum(AnaADC[c][:,Lim_ped[c]+10:np.argmax(Mean_AnaADC[c])+ranges[3]],axis=1);
# Reshape data for KMeans input
data_reshaped = Good_charge[np.where(Good_charge<30e3)].reshape(-1, 1)

# Specify the number of peaks you want to find
num_peaks = 6

# Use KMeans to find peaks
kmeans = KMeans(n_clusters=num_peaks)
kmeans.fit(data_reshaped)

# Get the centroids (peak locations)
peaks = kmeans.cluster_centers_.flatten()

h, edges=np.histogram(Good_charge[np.where(Good_charge<30e3)],bins=2000)

fig1=go.Figure();
fig1.add_trace(go.Bar(x=edges[:-1],y=h/np.max(h),name='Charge Histogram',opacity=1))
fig1.add_trace(go.Scatter(x=peaks, y=np.zeros_like(peaks), mode='markers', marker=dict(color='red', size=10), name='Peaks'))
#fig1.add_trace(go.Scatter(x=edges,y=adjusted_distribution,name='Fitted distribution'))
fig1.update_yaxes(title='Normalized counts')
fig1.update_xaxes(title='Charge (ADC counts * s)')
fig1.update_layout(title='Channel = '+str(c)+' Run = '+str(run),title_x=0.5,bargap = 0,template='simple_white')
fig1.show()

In [ ]:
print(peaks)